# Regularization Arcsinh


In [3]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 28.8 MB/s eta 0:00:00


In [4]:
from pathlib import Path
import json as _json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import RobustScaler
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pack_padded_sequence
from torch.optim.lr_scheduler import ReduceLROnPlateau
import optuna
from pathlib import Path as _P
import time as _time

SEED = 42
DATA_PATH = _P("Merged_Dataset_yoy.csv")
PREDICTION_TARGETS = ["EBITDA", "Net_Income", "ROA"]
DEFAULT_LOOKBACK_DAYS = 365

MAX_EPOCHS = 100
PATIENCE = 15
TEST_YEAR_CUTOFF = 2023
N_SPLITS = 5
OPTUNA_TRIALS = 30
N_FINAL_SEEDS = 5
N_FOLDS_FV = 5
WINSOR_CAP = 3.0
ARCSINH_C = 0.5
ARCSINH_CAP = float(np.arcsinh(WINSOR_CAP / ARCSINH_C))

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(max(1, torch.get_num_threads() // 2))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


## Data Processing

In [ ]:
def load_and_prepare_yoy_data(data_path: Path, lookback_days: int = 365):
    df = pd.read_csv(data_path)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values(["Company", "Date"]).reset_index(drop=True)
    df["year"] = df["Date"].dt.year

    df["Close_logret"] = df.groupby("Company")["Close"].transform(lambda x: np.log(x / x.shift(1)))
    df["Volume_log"] = np.log1p(df["Volume"].clip(lower=0))

    df["has_targets"] = df[PREDICTION_TARGETS].notna().any(axis=1)
    year_end_records = df[df["has_targets"]].copy()

    prior_static_cols = [
        "Prior_EBITDA_SL", "Prior_NI_SL", "Prior_ROA",
        "Leverage", "Cash_Ratio", "Size_SL",
        "Prior_Net_Sales_SL", "Prior_OpEx_SL", "Prior_FCF_Per_Share_SL",
    ]
    current_year_cols = [
        "Total_Assets", "Total_Debt", "Cash",
        "Net_Sales", "Operating_Expenses", "FCF_Per_Share",
    ]
    static_input_cols = ["MC_MIB", "MC_MID", "MC_SMALL", "Sector"]
    exclude_cols = ({"Date", "Company", "year", "has_targets"}
                    | set(PREDICTION_TARGETS) | set(prior_static_cols)
                    | set(current_year_cols) | set(static_input_cols))
    feature_cols = [col for col in df.columns
                    if col not in exclude_cols and pd.api.types.is_numeric_dtype(df[col])]

    sequences = []
    for company in df["Company"].unique():
        company_records = year_end_records[year_end_records["Company"] == company].copy()
        company_data = df[df["Company"] == company].copy()

        for _, year_end_row in company_records.iterrows():
            year = int(year_end_row["year"])
            year_end_date = year_end_row["Date"]

            prior_year_records = year_end_records[
                (year_end_records["Company"] == company)
                & (year_end_records["year"] == year - 1)
            ]
            if prior_year_records.empty:
                continue

            prior_row = prior_year_records.iloc[0]
            window_start = year_end_date - pd.Timedelta(days=lookback_days)
            window = company_data[
                (company_data["Date"] > window_start) & (company_data["Date"] <= year_end_date)
            ].copy()

            if len(window) < 200:
                continue

            window[feature_cols] = window[feature_cols].ffill().bfill()
            if window[feature_cols].isna().any().any():
                continue

            raw_window = window[feature_cols].to_numpy(dtype=np.float32)

            static_cols = ["MC_MIB", "MC_MID", "MC_SMALL"] + prior_static_cols
            static_data = window[static_cols].iloc[-1].to_numpy(dtype=np.float32)
            sector_str = str(window["Sector"].iloc[-1])

            sequences.append({
                "company": company, "year": year, "year_end_date": year_end_date,
                "window_data": raw_window,
                "static_data": static_data,
                "sector": sector_str,
                "window_dates": window["Date"].to_numpy(),
                "window_days": (window["Date"] - window["Date"].iloc[0]).dt.days.to_numpy(),
                **{f"current_{t.lower()}": year_end_row[t] for t in PREDICTION_TARGETS},
                **{f"prior_{t.lower()}": prior_row[t] for t in PREDICTION_TARGETS},
            })

    data_records = []
    for seq in sequences:
        for target_name in PREDICTION_TARGETS:
            current_val = seq[f"current_{target_name.lower()}"]
            prior_val = seq[f"prior_{target_name.lower()}"]
            if pd.isna(current_val) or pd.isna(prior_val):
                continue

            label_value = (current_val - prior_val) / (np.abs(prior_val) + 1e-8)

            data_records.append({
                "company": seq["company"], "year": seq["year"],
                "year_end_date": seq["year_end_date"],
                "target": target_name, "label_value": label_value,
                "window_data": seq["window_data"],
                "static_data": seq["static_data"],
                "sector": seq["sector"],
                "window_dates": seq["window_dates"],
                "window_days": seq["window_days"],
            })

    return pd.DataFrame(data_records), feature_cols

data_df, feature_cols = load_and_prepare_yoy_data(DATA_PATH, lookback_days=DEFAULT_LOOKBACK_DAYS)
print(f"Total rows: {len(data_df)}")
print(f"feature_cols ({len(feature_cols)}): {feature_cols}")

Total rows: 6715
feature_cols (34): ['Close', 'High', 'Low', 'Open', 'Volume', 'DE10YT_Yield', 'EURUSD_Close', 'EURUSD_Volume', 'Euribor_3M', 'FTMIB_Close', 'FTMIB_Volume', 'GVZ_Close', 'IT10YT_Yield', 'IT_Inflation', 'IT_GDP', 'IT_Unemployment', 'Brent_Close', 'Brent_Volume', 'OVX_Close', 'TTF_Gas_Close', 'VIX_Close', 'Gold_Close', 'T5YIE', 'YC__level', 'YC__slope', 'YC__curv', 'PCOPPUSDM', 'PALUMUSDM', 'NFCI', 'ff_Mkt-RF', 'ff_HML', 'ff_Mom', 'Close_logret', 'Volume_log']


## Pipeline Setup: Arcsinh c=0.5, Winsor +/-3, Sector + Prior Statics, 5-Fold TSCV

In [ ]:
def temporal_kfold_split(df: pd.DataFrame, n_splits: int = N_SPLITS, test_year_cutoff: int = TEST_YEAR_CUTOFF):
    if df.empty:
        raise ValueError("The dataset is empty.")
    available_years = sorted(int(y) for y in df["year"].unique())
    pre_test_years = [y for y in available_years if y <= test_year_cutoff]
    print(f"Available years: {available_years}")
    if len(pre_test_years) < n_splits + 1:
        raise ValueError(f"Not enough pre-test years ({len(pre_test_years)}) for {n_splits} folds.")
    val_years = pre_test_years[-n_splits:]
    test_set = df[df["year"] > test_year_cutoff].copy()
    print(f"Test: > {test_year_cutoff} ({len(test_set)} samples)")
    print(f"Temporal k-fold ({n_splits} folds, expanding window). Val years: {val_years}")
    folds = []
    for fold_idx, vy in enumerate(val_years, start=1):
        train_set = df[df["year"] < vy].copy()
        val_set = df[df["year"] == vy].copy()
        if train_set.empty or val_set.empty:
            print(f"WARNING: fold {fold_idx} (val {vy}) is empty. Skipping.")
            continue
        folds.append((train_set, val_set, vy, fold_idx))
        print(f"  Fold {fold_idx}: train <= {vy-1} ({len(train_set)}), val = {vy} ({len(val_set)})")
    if not folds:
        raise ValueError("No valid folds produced.")
    return folds, test_set

def split_to_single(df: pd.DataFrame, test_year_cutoff: int = TEST_YEAR_CUTOFF):
    folds, test_set = temporal_kfold_split(df, n_splits=1, test_year_cutoff=test_year_cutoff)
    train_set, val_set, _, _ = folds[-1]
    return train_set, val_set, test_set

def make_robust_scaler(train_df: pd.DataFrame, feature_indices):
    sc = RobustScaler()
    train_windows = np.vstack([row[:, feature_indices] for row in train_df["window_data"]])
    sc.fit(train_windows)
    return sc

def fit_static_scaler(mtl_df, new_slice):
    arr = np.vstack(mtl_df["static_data"].values).astype(np.float32)
    return RobustScaler().fit(arr[:, new_slice])

def apply_static_scaler(mtl_df, sc, new_slice):
    df = mtl_df.copy().reset_index(drop=True)
    arr = np.vstack(df["static_data"].values).astype(np.float32)
    arr[:, new_slice] = sc.transform(arr[:, new_slice]).astype(np.float32)
    df["static_data"] = [arr[i] for i in range(len(arr))]
    return df

SECTOR_LIST = sorted(data_df["sector"].dropna().unique())
SECTOR_TO_IDX = {s: i for i, s in enumerate(SECTOR_LIST)}
N_SECTORS = len(SECTOR_LIST)
print(f"Sectors ({N_SECTORS}): {SECTOR_LIST}")

def sector_onehot(sector_str):
    v = np.zeros(N_SECTORS, dtype=np.float32)
    if sector_str in SECTOR_TO_IDX:
        v[SECTOR_TO_IDX[sector_str]] = 1.0
    return v

def build_full_static(static_data_arr, sector_str):
    mc_and_prior = np.asarray(static_data_arr, dtype=np.float32)  # 12 dims (3 MC + 9 Prior_*)
    sector_oh = sector_onehot(sector_str)                          # N_SECTORS dims
    return np.concatenate([mc_and_prior[:3], sector_oh, mc_and_prior[3:]]).astype(np.float32)

# Drop samples whose prior-year fundamentals (or MC dummy) are NaN.
static_nan_mask = data_df["static_data"].apply(
    lambda a: bool(np.isnan(np.asarray(a, dtype=float)).any())
)
n_drop_missing = int(static_nan_mask.sum())
data_df = data_df.loc[~static_nan_mask].copy().reset_index(drop=True)
print(f"Dropped {n_drop_missing} samples missing prior-year fundamentals; {len(data_df)} remaining.")

data_df["static_data"] = data_df.apply(
    lambda r: build_full_static(r["static_data"], r["sector"]), axis=1
)
STATIC_DIM_IMP = 3 + N_SECTORS + 9  # MC(3) + sector(N_SECTORS) + prior(9)
NEW_STATIC_SLICE = slice(3 + N_SECTORS, 3 + N_SECTORS + 9)  # scale only the prior block
print(f"STATIC_DIM_IMP={STATIC_DIM_IMP} (MC=3, sector={N_SECTORS}, prior=9)")
print(f"NEW_STATIC_SLICE = {NEW_STATIC_SLICE.start}:{NEW_STATIC_SLICE.stop} (only prior gets RobustScaler)")

def extract_mtl_df(df, new_slice=NEW_STATIC_SLICE):
    mtl_records = []
    for (company, year), group in df.groupby(["company", "year"]):
        row = group.iloc[0]
        label_value = np.full(len(PREDICTION_TARGETS), np.nan, dtype=np.float32)
        for i, t in enumerate(PREDICTION_TARGETS):
            t_row = group[group["target"] == t]
            if not t_row.empty:
                label_value[i] = t_row.iloc[0]["label_value"]
        mtl_records.append({
            "company": company, "year": year, "year_end_date": row["year_end_date"],
            "window_data": row["window_data"],
            "static_data": row["static_data"],
            "window_dates": row["window_dates"],
            "window_days": row["window_days"],
            "label_value": label_value,
        })
    return pd.DataFrame(mtl_records)

# Winsor +/-3 + arcsinh c=0.5 to clip tails in real space
raw_ratio = data_df["label_value"].to_numpy(dtype=np.float32)
capped = np.clip(raw_ratio, -WINSOR_CAP, WINSOR_CAP)
data_df["label_value"] = np.arcsinh(capped / ARCSINH_C).astype(np.float32)
n_capped = int(np.sum(np.abs(raw_ratio) > WINSOR_CAP))
print(f"Winsor cap: {n_capped}/{len(raw_ratio)} ({100.0*n_capped/len(raw_ratio):.2f}%) at +/-{WINSOR_CAP}")

# Build k-fold for HPO + single-split for final test
folds, test_data = temporal_kfold_split(data_df, n_splits=N_SPLITS, test_year_cutoff=TEST_YEAR_CUTOFF)
train_data, val_data, _ = split_to_single(data_df, test_year_cutoff=TEST_YEAR_CUTOFF)

mtl_train_data_raw = extract_mtl_df(train_data)
mtl_val_data_raw   = extract_mtl_df(val_data)
mtl_test_data      = extract_mtl_df(test_data)

train_per_year = mtl_train_data_raw["year"].value_counts().sort_index()
test_per_year  = mtl_test_data["year"].value_counts().sort_index()
print(f"Train: {len(mtl_train_data_raw)} ({dict(train_per_year)})")
print(f"Test:  {len(mtl_test_data)} ({dict(test_per_year)})")

Sectors (10): ['Basic Materials', 'Consumer Cyclicals', 'Consumer Non-Cyclicals', 'Energy', 'Financials', 'Healthcare', 'Industrials', 'Real Estate', 'Technology', 'Utilities']
Dropped 970 samples missing prior-year fundamentals; 5745 remaining.
STATIC_DIM_IMP=22 (MC=3, sector=10, prior=9)
NEW_STATIC_SLICE = 13:22 (only prior gets RobustScaler)
Winsor cap: 493/5745 (8.58%) at +/-3.0
Available years: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Test: > 2023 (919 samples)
Temporal k-fold (5 folds, expanding window). Val years: [2019, 2020, 2021, 2022, 2023]
  Fold 1: train <= 2018 (2739), val = 2019 (376)
  Fold 2: train <= 2019 (3115), val = 2020 (395)
  Fold 3: train <= 2020 (3510), val = 2021 (419)
  Fold 4: train <= 2021 (3929), val = 2022 (438)
  Fold 5: train <= 2022 (4367), val = 2023 (459)
Available years: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Test: > 2023 (919 samples)


## Frequencies feature selection for MultiFreqLSTM

In [ ]:
DAILY_FEATURES = [
    'Close_logret', 'Volume_log', 'DE10YT_Yield', 'IT10YT_Yield',
    'EURUSD_Close', 'EURUSD_Volume', 'FTMIB_Close', 'FTMIB_Volume',
    'GVZ_Close', 'Brent_Close', 'Brent_Volume', 'OVX_Close', 'VIX_Close',
    'Gold_Close', 'YC__level', 'YC__slope', 'YC__curv', 'ff_HML', 'ff_Mom',
    'TTF_Gas_Close', 'NFCI',
]
MONTHLY_FEATURES = ['Euribor_3M', 'IT_Inflation', 'IT_Unemployment', 'PCOPPUSDM', 'PALUMUSDM']
QUARTERLY_FEATURES = ['IT_GDP']

print(f"Daily ({len(DAILY_FEATURES)}): {DAILY_FEATURES}")
print(f"Monthly ({len(MONTHLY_FEATURES)}): {MONTHLY_FEATURES}")
print(f"Quarterly ({len(QUARTERLY_FEATURES)}): {QUARTERLY_FEATURES}")

daily_indices     = [feature_cols.index(f) for f in DAILY_FEATURES]
monthly_indices   = [feature_cols.index(f) for f in MONTHLY_FEATURES]
quarterly_indices = [feature_cols.index(f) for f in QUARTERLY_FEATURES]

Daily (21): ['Close_logret', 'Volume_log', 'DE10YT_Yield', 'IT10YT_Yield', 'EURUSD_Close', 'EURUSD_Volume', 'FTMIB_Close', 'FTMIB_Volume', 'GVZ_Close', 'Brent_Close', 'Brent_Volume', 'OVX_Close', 'VIX_Close', 'Gold_Close', 'YC__level', 'YC__slope', 'YC__curv', 'ff_HML', 'ff_Mom', 'TTF_Gas_Close', 'NFCI']
Monthly (5): ['Euribor_3M', 'IT_Inflation', 'IT_Unemployment', 'PCOPPUSDM', 'PALUMUSDM']
Quarterly (1): ['IT_GDP']


## Model Definitions

In [ ]:
class ShallowLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_targets=3, dropout=0.2, static_dim=0):
        super().__init__()
        self.static_dim = static_dim
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(p=dropout)
        self.head = nn.Linear(hidden_size + static_dim, num_targets)

    def forward(self, x, lengths, static=None):
        packed_x = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=True)
        _, (hn, _) = self.lstm(packed_x)
        hidden = self.dropout(hn[-1])
        if self.static_dim > 0 and static is not None:
            hidden = torch.cat([hidden, static.to(hidden.device, hidden.dtype)], dim=-1)
        logits = self.head(hidden)
        if logits.size(-1) == 1:
            logits = logits.squeeze(-1)
        return logits

class StackedLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1, num_targets=3,
                 dropout=0.2, static_dim=0):
        super().__init__()
        self.static_dim = static_dim
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers,
                            batch_first=True, dropout=0.0)
        self.dropout = nn.Dropout(p=dropout)
        self.head = nn.Linear(hidden_size + static_dim, num_targets)

    def forward(self, x, lengths, static=None):
        packed_x = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=True)
        _, (hn, _) = self.lstm(packed_x)
        hidden = self.dropout(hn[-1])
        if self.static_dim > 0 and static is not None:
            hidden = torch.cat([hidden, static.to(hidden.device, hidden.dtype)], dim=-1)
        logits = self.head(hidden)
        if logits.size(-1) == 1:
            logits = logits.squeeze(-1)
        return logits

class MultiFreqLSTM_NoQuarterly(nn.Module):
    """Two-stream variant of MultiFreqLSTM: daily + monthly only.
    Quarterly IT_GDP is folded into the monthly stream (forward-filled at
    monthly cadence), so no separate quarterly MLP is used."""
    def __init__(self, n_daily, n_monthly, n_static,
                 d_daily=64, d_monthly=32,
                 n_layers_daily=2, n_layers_monthly=1,
                 num_targets=3, dropout=0.1):
        super().__init__()
        self.daily = nn.LSTM(n_daily, d_daily, num_layers=n_layers_daily,
                            batch_first=True, dropout=0.0)
        self.monthly = nn.LSTM(n_monthly, d_monthly, num_layers=n_layers_monthly,
                              batch_first=True, dropout=0.0)
        self.merge_dim = d_daily + d_monthly + n_static
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(self.merge_dim, num_targets)
        self._squeeze = (num_targets == 1)

    def forward(self, daily, monthly, quarterly, lengths, static):
        packed_d = pack_padded_sequence(daily, lengths.cpu(),
                                        batch_first=True, enforce_sorted=True)
        _, (h_d, _) = self.daily(packed_d)
        h_d = h_d[-1]

        B = monthly.size(0)
        lengths_m = torch.full((B,), 12, dtype=torch.long, device=monthly.device)
        packed_m = pack_padded_sequence(monthly, lengths_m.cpu(),
                                        batch_first=True, enforce_sorted=False)
        _, (h_m, _) = self.monthly(packed_m)
        h_m = h_m[-1]

        h = torch.cat([h_d, h_m, static.to(h_d.device, h_d.dtype)], dim=-1)
        h = self.dropout(h)
        logits = self.head(h)
        if self._squeeze:
            logits = logits.squeeze(-1)
        return logits

selected_indices = [feature_cols.index(f) for f in (DAILY_FEATURES + MONTHLY_FEATURES + QUARTERLY_FEATURES)]

print("Models defined: ShallowLSTM, StackedLSTM, MultiFreqLSTM_NoQuarterly")
def init_lstm_submodule(model, head_std=1e-3):
    """Weight init for Shallow/Stacked LSTM and MultiFreqLSTM_NoQuarterly.
    - Xavier-uniform on every LSTM weight.
    - LSTM biases: zero, except the forget-gate slice, which is set to 1.0
      (PyTorch default for stable LSTM training).
    - Small normal head (std=head_std) + zero bias so initial output ~ 0.
    """
    for m in model.modules():
        cls_name = type(m).__name__
        if cls_name == "LSTM":
            for name, p in m.named_parameters():
                if "weight" in name:
                    nn.init.xavier_uniform_(p)
                elif "bias" in name:
                    nn.init.zeros_(p)
                    n = p.numel() // 4
                    with torch.no_grad():
                        p[n:2*n].fill_(1.0)  # forget-gate bias = 1
    if hasattr(model, "head"):
        nn.init.normal_(model.head.weight, mean=0.0, std=head_std)
        nn.init.zeros_(model.head.bias)


Models defined: ShallowLSTM, StackedLSTM, MultiFreqLSTM_NoQuarterly


## Datasets & DataLoaders

In [ ]:
def aggregate_to_cadence(window_dates, window_days, raw_window, target_indices, n_steps):
    """Aggregate a raw window to `n_steps` evenly-spaced bins (by day of year).
    raw_window: (T, F) array; target_indices: list of feature column indices to keep.
    Returns (n_steps, len(target_indices)) array, with NaN if no observations.
    """
    T = raw_window.shape[0]
    if T < 2:
        return np.full((n_steps, len(target_indices)), np.nan, dtype=np.float32)
    last_day = float(window_days[-1])
    if last_day <= 0:
        return np.full((n_steps, len(target_indices)), np.nan, dtype=np.float32)
    bin_size = last_day / n_steps
    out = np.full((n_steps, len(target_indices)), np.nan, dtype=np.float32)
    if len(target_indices) == 0:
        return out
    # Assign each timestep to its bin
    bin_idx = np.minimum((window_days / bin_size).astype(int), n_steps - 1)
    for f_i, f_idx in enumerate(target_indices):
        col = raw_window[:, f_idx]
        for b in range(n_steps):
            mask = bin_idx == b
            if mask.any():
                vals = col[mask]
                vals = vals[~np.isnan(vals)]
                if len(vals) > 0:
                    out[b, f_i] = vals[-1]  # take last non-NaN value (forward-fill)
    return out

def build_multifreq_window(raw_window, window_days, daily_idx, monthly_idx, quarterly_idx,
                            n_daily=252, n_monthly=12, n_quarterly=4):
    """Aggregate the raw window to the 3 cadences."""
    daily    = aggregate_to_cadence(None, window_days, raw_window, daily_idx,     n_daily)
    monthly  = aggregate_to_cadence(None, window_days, raw_window, monthly_idx,   n_monthly)
    quarterly = aggregate_to_cadence(None, window_days, raw_window, quarterly_idx, n_quarterly)
    # Forward-fill along the time axis (within stream)
    for arr in (daily, monthly, quarterly):
        for f in range(arr.shape[1]):
            last = np.nan
            for t in range(arr.shape[0]):
                if np.isnan(arr[t, f]):
                    arr[t, f] = last
                else:
                    last = arr[t, f]
            # Backward-fill leading NaNs
            if np.isnan(arr[0, f]):
                first_valid = np.nan
                for t in range(arr.shape[0]):
                    if not np.isnan(arr[t, f]):
                        first_valid = arr[t, f]
                        break
                arr[:, f] = first_valid
    # Replace any remaining NaN with 0
    daily    = np.nan_to_num(daily,    nan=0.0).astype(np.float32)
    monthly  = np.nan_to_num(monthly,  nan=0.0).astype(np.float32)
    quarterly = np.nan_to_num(quarterly, nan=0.0).astype(np.float32)
    return daily, monthly, quarterly

class YoYDatasetMTL(Dataset):
    def __init__(self, data_df, scaler, feature_indices, window_size=None):
        self.data_df = data_df.reset_index(drop=True)
        self.scaler = scaler
        self.feature_indices = feature_indices
        self.window_size = window_size

    def __len__(self):
        return len(self.data_df)

    def __getitem__(self, idx):
        row = self.data_df.iloc[idx]
        window = row["window_data"][:, self.feature_indices].copy()
        if self.window_size is not None:
            window = window[-self.window_size:]
        if self.scaler is not None:
            window = self.scaler.transform(window)
            # Sanitize: RobustScaler can produce NaN/Inf
            window = np.nan_to_num(window, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        static = row["static_data"].astype(np.float32)
        static = np.nan_to_num(static, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        target = row["label_value"]
        mask = ~np.isnan(target)
        target_clean = np.nan_to_num(target, nan=0.0)
        return (
            torch.from_numpy(window),
            torch.from_numpy(static),
            torch.tensor(target_clean, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.bool),
            len(window),
        )

def collate_fn_mtl(batch):
    batch.sort(key=lambda x: x[4], reverse=True)
    sequences = [x[0] for x in batch]
    statics = torch.stack([x[1] for x in batch])
    targets = torch.stack([x[2] for x in batch])
    masks = torch.stack([x[3] for x in batch])
    lengths = torch.tensor([x[4] for x in batch])
    padded_seqs = torch.nn.utils.rnn.pad_sequence(sequences, batch_first=True)
    return padded_seqs, targets, masks, statics, lengths

class MultiFreqDataset(Dataset):
    def __init__(self, mtl_df, daily_scaler, monthly_scaler, quarterly_scaler, stat_scaler,
                 daily_idx, monthly_idx, quarterly_idx, n_daily=252, n_monthly=12, n_quarterly=4):
        self.mtl_df = mtl_df.reset_index(drop=True)
        self.daily_scaler = daily_scaler
        self.monthly_scaler = monthly_scaler
        self.quarterly_scaler = quarterly_scaler
        self.stat_scaler = stat_scaler
        self.daily_idx = daily_idx
        self.monthly_idx = monthly_idx
        self.quarterly_idx = quarterly_idx
        self.n_daily = n_daily
        self.n_monthly = n_monthly
        self.n_quarterly = n_quarterly

    def __len__(self):
        return len(self.mtl_df)

    def __getitem__(self, idx):
        row = self.mtl_df.iloc[idx]
        raw_window = row["window_data"]
        window_days = row["window_days"]
        d, m, q = build_multifreq_window(
            raw_window, window_days,
            self.daily_idx, self.monthly_idx, self.quarterly_idx,
            n_daily=self.n_daily, n_monthly=self.n_monthly, n_quarterly=self.n_quarterly,
        )
        if self.daily_scaler is not None:
            d = self.daily_scaler.transform(d)
            d = np.nan_to_num(d, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        if self.monthly_scaler is not None:
            m = self.monthly_scaler.transform(m)
            m = np.nan_to_num(m, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        if self.quarterly_scaler is not None:
            q = self.quarterly_scaler.transform(q)
            q = np.nan_to_num(q, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        if self.stat_scaler is not None:
            static = self.stat_scaler.transform(
                row["static_data"].reshape(1, -1)
            ).reshape(-1).astype(np.float32)
        else:
            static = row["static_data"].astype(np.float32)
        static = np.nan_to_num(static, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        target = row["label_value"]
        mask = ~np.isnan(target)
        target_clean = np.nan_to_num(target, nan=0.0)
        return (
            torch.from_numpy(d.astype(np.float32)),
            torch.from_numpy(m.astype(np.float32)),
            torch.from_numpy(q.astype(np.float32)),
            torch.from_numpy(static),
            torch.tensor(target_clean, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.bool),
            self.n_daily,
        )

def collate_fn_multifreq(batch):
    batch.sort(key=lambda x: x[6], reverse=True)
    ds = torch.stack([x[0] for x in batch])
    ms = torch.stack([x[1] for x in batch])
    qs = torch.stack([x[2] for x in batch])
    statics = torch.stack([x[3] for x in batch])
    targets = torch.stack([x[4] for x in batch])
    masks = torch.stack([x[5] for x in batch])
    lengths = torch.tensor([x[6] for x in batch])
    return ds, ms, qs, targets, masks, statics, lengths

## Training with Regularization setup

In [ ]:
def arcsinh_inverse(arr):
    return ARCSINH_C * np.sinh(arr)

def train_and_eval_mtl(model, train_loader, val_loader, optimizer,
                       trial=None, step_offset=0):
    """L1 loss, ES, LR scheduler, grad_clip=1.0, best_state checkpointing."""
    criterion = nn.L1Loss(reduction='none')
    scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)
    best_state = None
    best_metric = float("inf")
    epochs_no_improve = 0
    best_epoch = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            bx, by, bmask, bstatic, lengths = batch
            bx, by, bmask, bstatic = bx.to(DEVICE), by.to(DEVICE), bmask.to(DEVICE), bstatic.to(DEVICE)
            logits = model(bx, lengths, bstatic)
            loss_m = criterion(logits, by)
            loss = loss_m[bmask].mean() if loss_m[bmask].numel() > 0 else 0 * loss_m.sum()
            if isinstance(loss, torch.Tensor) and loss.requires_grad:
                if not torch.isfinite(loss):
                    continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        model.eval()
        all_preds, all_targets, all_masks = [], [], []
        with torch.no_grad():
            for batch in val_loader:
                bx, by, bmask, bstatic, lengths = batch
                bx, bstatic = bx.to(DEVICE), bstatic.to(DEVICE)
                logits = model(bx, lengths, bstatic)
                all_masks.append(bmask.numpy())
                all_preds.append(logits.cpu().numpy())
                all_targets.append(by.numpy())
        if not all_preds:
            break
        y_pred = np.vstack(all_preds)
        y_true = np.vstack(all_targets)
        mask = np.vstack(all_masks)
        valid_sum, count = 0.0, 0
        for i in range(len(PREDICTION_TARGETS)):
            m = mask[:, i]
            if m.sum() == 0:
                continue
            valid_sum += np.mean(np.abs(arcsinh_inverse(y_pred[m, i]) - arcsinh_inverse(y_true[m, i])))
            count += 1
        current_metric = valid_sum / max(1, count)
        scheduler.step(current_metric)

        if np.isfinite(current_metric) and current_metric < best_metric:
            best_metric = current_metric
            best_state = {n: t.detach().cpu().clone() for n, t in model.state_dict().items()}
            best_epoch = epoch
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        if trial is not None:
            trial.report(current_metric, step_offset + epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()
        if epochs_no_improve >= PATIENCE:
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_metric, best_epoch

def evaluate_mtl_loader(model, loader):
    model.eval()
    all_preds, all_targets, all_masks = [], [], []
    with torch.no_grad():
        for bx, by, bmask, bstatic, lengths in loader:
            logits = model(bx.to(DEVICE), lengths, bstatic.to(DEVICE))
            all_preds.append(logits.cpu().numpy())
            all_targets.append(by.numpy())
            all_masks.append(bmask.numpy())
    if not all_preds:
        return None
    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)
    mask = np.vstack(all_masks).astype(bool)
    y_pred_r = arcsinh_inverse(y_pred)
    y_true_r = arcsinh_inverse(y_true)
    res = {}
    all_err = []
    for i, t in enumerate(PREDICTION_TARGETS):
        m = mask[:, i]
        if m.sum() == 0:
            continue
        res[t] = float(np.mean(np.abs(y_pred_r[m, i] - y_true_r[m, i])))
        all_err.extend(np.abs(y_pred_r[m, i] - y_true_r[m, i]).tolist())
    res["Overall"] = float(np.mean(all_err)) if all_err else None
    return res

print("MTL training/eval helpers ready.")

def collect_predictions_mtl(model, loader):
    """Run model on loader, return (predictions, targets) in real space."""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for bx, by, bmask, bstatic, lengths in loader:
            logits = model(bx.to(DEVICE), lengths, bstatic.to(DEVICE))
            preds.append(logits.cpu().numpy())
            targets.append(by.numpy())
    yp = np.vstack(preds); yt = np.vstack(targets)
    yp_r = arcsinh_inverse(yp); yt_r = arcsinh_inverse(yt)
    return yp_r, yt_r

def collect_predictions_mf(model, loader):
    """MultiFreq version: run model on loader, return (predictions, targets) in real space."""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for bd, bm, bq, by, bmask, bstatic, lengths in loader:
            bd = bd.to(DEVICE); bm = bm.to(DEVICE); bq = bq.to(DEVICE)
            bstatic = bstatic.to(DEVICE)
            logits = model(bd, bm, bq, lengths, bstatic)
            preds.append(logits.cpu().numpy())
            targets.append(by.numpy())
    yp = np.vstack(preds); yt = np.vstack(targets)
    yp_r = arcsinh_inverse(yp); yt_r = arcsinh_inverse(yt)
    return yp_r, yt_r

MTL training/eval helpers ready.


## MultiFreq specific training/eval helpers

In [ ]:
def train_and_eval_mf(model, train_loader, val_loader, optimizer,
                      trial=None, step_offset=0):
    criterion = nn.L1Loss(reduction='none')
    scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)
    best_state = None
    best_metric = float("inf")
    best_epoch = 0
    epochs_no_improve = 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for bd, bm, bq, by, bmask, bstatic, lengths in train_loader:
            bd = bd.to(DEVICE); bm = bm.to(DEVICE); bq = bq.to(DEVICE)
            by = by.to(DEVICE); bmask = bmask.to(DEVICE); bstatic = bstatic.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(bd, bm, bq, lengths, bstatic)
            loss_m = criterion(logits, by)
            loss = loss_m[bmask].mean() if loss_m[bmask].numel() > 0 else 0 * loss_m.sum()
            if isinstance(loss, torch.Tensor) and loss.requires_grad:
                if not torch.isfinite(loss):
                    continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
        model.eval()
        all_preds, all_targets, all_masks = [], [], []
        with torch.no_grad():
            for bd, bm, bq, by, bmask, bstatic, lengths in val_loader:
                bd = bd.to(DEVICE); bm = bm.to(DEVICE); bq = bq.to(DEVICE)
                bstatic = bstatic.to(DEVICE)
                logits = model(bd, bm, bq, lengths, bstatic)
                all_preds.append(logits.cpu().numpy())
                all_targets.append(by.numpy())
                all_masks.append(bmask.numpy())
        if not all_preds:
            break
        y_pred = np.vstack(all_preds)
        y_true = np.vstack(all_targets)
        mask = np.vstack(all_masks).astype(bool)
        y_pred_r = arcsinh_inverse(y_pred)
        y_true_r = arcsinh_inverse(y_true)
        valid_sum, count = 0.0, 0
        for i in range(len(PREDICTION_TARGETS)):
            m = mask[:, i]
            if m.sum() == 0:
                continue
            valid_sum += np.mean(np.abs(y_pred_r[m, i] - y_true_r[m, i]))
            count += 1
        current_metric = valid_sum / max(1, count)
        scheduler.step(current_metric)

        if np.isfinite(current_metric) and current_metric < best_metric:
            best_metric = current_metric
            best_state = {n: t.detach().cpu().clone() for n, t in model.state_dict().items()}
            best_epoch = epoch
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        if trial is not None:
            trial.report(current_metric, step_offset + epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()
        if epochs_no_improve >= PATIENCE:
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_metric, best_epoch

def evaluate_mf_loader(model, loader):
    model.eval()
    all_preds, all_targets, all_masks = [], [], []
    with torch.no_grad():
        for bd, bm, bq, by, bmask, bstatic, lengths in loader:
            bd = bd.to(DEVICE); bm = bm.to(DEVICE); bq = bq.to(DEVICE)
            bstatic = bstatic.to(DEVICE)
            logits = model(bd, bm, bq, lengths, bstatic)
            all_preds.append(logits.cpu().numpy())
            all_targets.append(by.numpy())
            all_masks.append(bmask.numpy())
    if not all_preds:
        return None
    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)
    mask = np.vstack(all_masks).astype(bool)
    y_pred_r = arcsinh_inverse(y_pred)
    y_true_r = arcsinh_inverse(y_true)
    res = {}
    all_err = []
    for i, t in enumerate(PREDICTION_TARGETS):
        m = mask[:, i]
        if m.sum() == 0:
            continue
        res[t] = float(np.mean(np.abs(y_pred_r[m, i] - y_true_r[m, i])))
        all_err.extend(np.abs(y_pred_r[m, i] - y_true_r[m, i]).tolist())
    res["Overall"] = float(np.mean(all_err)) if all_err else None
    return res

## Loader builders per model

In [ ]:
def build_mtl_loaders_for_fold(tr_df, va_df, params, feature_indices, static_dim):
    """Build train/val/test loaders for MTL (Shallow or Stacked LSTM).
    Returns (model, train_loader, val_loader, test_loader, scaler, stat_scaler)."""
    tr_mtl = extract_mtl_df(tr_df)
    va_mtl = extract_mtl_df(va_df)
    te_mtl = extract_mtl_df(test_data)
    temp_sc = make_robust_scaler(tr_mtl, feature_indices)
    stat_sc = fit_static_scaler(tr_mtl, NEW_STATIC_SLICE)
    tr_mtl_s = apply_static_scaler(tr_mtl, stat_sc, NEW_STATIC_SLICE)
    va_mtl_s = apply_static_scaler(va_mtl, stat_sc, NEW_STATIC_SLICE)
    te_mtl_s = apply_static_scaler(te_mtl, stat_sc, NEW_STATIC_SLICE)
    tr_ds = YoYDatasetMTL(tr_mtl_s, temp_sc, feature_indices)
    va_ds = YoYDatasetMTL(va_mtl_s, temp_sc, feature_indices)
    te_ds = YoYDatasetMTL(te_mtl_s, temp_sc, feature_indices)
    tr_loader = DataLoader(tr_ds, batch_size=params['batch_size'], shuffle=True,  collate_fn=collate_fn_mtl)
    va_loader = DataLoader(va_ds, batch_size=params['batch_size'], shuffle=False, collate_fn=collate_fn_mtl)
    te_loader = DataLoader(te_ds, batch_size=params['batch_size'], shuffle=False, collate_fn=collate_fn_mtl)
    return tr_loader, va_loader, te_loader, temp_sc, stat_sc

def make_shallow_mtl(tr_df, va_df, params, feature_indices):
    tr_loader, va_loader, te_loader, _, _ = build_mtl_loaders_for_fold(
        tr_df, va_df, params, feature_indices, STATIC_DIM_IMP)
    model = ShallowLSTM(
        input_size=len(feature_indices),
        hidden_size=params['hidden_size'],
        num_targets=len(PREDICTION_TARGETS),
        dropout=params.get('dropout', 0.2),
        static_dim=STATIC_DIM_IMP,
    ).to(DEVICE)
    init_lstm_submodule(model)
    return model, tr_loader, va_loader, te_loader

def make_stacked_mtl(tr_df, va_df, params, feature_indices, num_layers=4):
    tr_loader, va_loader, te_loader, _, _ = build_mtl_loaders_for_fold(
        tr_df, va_df, params, feature_indices, STATIC_DIM_IMP)
    model = StackedLSTM(
        input_size=len(feature_indices),
        hidden_size=params['hidden_size'],
        num_layers=num_layers,
        num_targets=len(PREDICTION_TARGETS),
        dropout=params.get('dropout', 0.2),
        static_dim=STATIC_DIM_IMP,
    ).to(DEVICE)
    init_lstm_submodule(model)
    return model, tr_loader, va_loader, te_loader

# Optuna Search

In [ ]:
def get_hpo_objective_mtl(make_model_fn, feature_indices, n_trials=OPTUNA_TRIALS):
    fold_local = folds[-1]
    tr_df, va_df, val_year, _ = fold_local
    tr_mtl = extract_mtl_df(tr_df)
    va_mtl = extract_mtl_df(va_df)
    temp_sc = make_robust_scaler(tr_mtl, feature_indices)
    stat_sc = fit_static_scaler(tr_mtl, NEW_STATIC_SLICE)
    tr_mtl_s = apply_static_scaler(tr_mtl, stat_sc, NEW_STATIC_SLICE)
    va_mtl_s = apply_static_scaler(va_mtl, stat_sc, NEW_STATIC_SLICE)

    def objective(trial):
        hidden_size = trial.suggest_categorical("hidden_size", [32, 64, 128])
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
        dropout = trial.suggest_float("dropout", 0.0, 0.5)
        params = dict(hidden_size=hidden_size, lr=lr, batch_size=batch_size,
                      weight_decay=weight_decay, dropout=dropout)
        tr_ds = YoYDatasetMTL(tr_mtl_s, temp_sc, feature_indices)
        va_ds = YoYDatasetMTL(va_mtl_s, temp_sc, feature_indices)
        tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,  collate_fn=collate_fn_mtl)
        va_loader = DataLoader(va_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_mtl)
        model = make_model_fn(params, feature_indices)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        try:
            _, best_metric, _ = train_and_eval_mtl(
                model, tr_loader, va_loader, optimizer,
                trial=trial, step_offset=0,
            )
        except optuna.exceptions.TrialPruned:
            raise
        return best_metric
    return objective

def run_optuna(study_name, objective_fn, n_trials=OPTUNA_TRIALS, seed_trials=None):
    pruner = optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=15)
    study = optuna.create_study(direction="minimize", study_name=study_name, pruner=pruner)
    for t in (seed_trials or []):
        study.enqueue_trial(t)
    study.optimize(objective_fn, n_trials=n_trials)
    return study

## Model A: ShallowLSTM

In [ ]:
selected_indices = [feature_cols.index(f) for f in (DAILY_FEATURES + MONTHLY_FEATURES + QUARTERLY_FEATURES)]
print(f"Using {len(selected_indices)} features for Shallow/Stacked LSTM input.")

def make_shallow_fn(params, fi):
    model = ShallowLSTM(len(selected_indices), params['hidden_size'], len(PREDICTION_TARGETS),
                        params.get('dropout', 0.2), static_dim=STATIC_DIM_IMP).to(DEVICE)
    init_lstm_submodule(model)
    return model

_t0 = _time.time()
study_A = run_optuna(
    "shallow_reg",
    get_hpo_objective_mtl(make_shallow_fn, selected_indices),
    n_trials=OPTUNA_TRIALS,
    seed_trials=[{
        'hidden_size': 128, 'lr': 0.00859001743156823, 'batch_size': 16,
        'weight_decay': 7.262266887516058e-05, 'dropout': 0.05642680822122102,
    }],
)
print(f"\nShallowLSTM best val MAE: {study_A.best_value:.4f}")
print(f"Best params: {study_A.best_trial.params}")
print(f"Optuna time: {(_time.time()-_t0)/60:.1f} min")

Using 27 features for Shallow/Stacked LSTM input.


[I 2026-08-10 01:38:36,070] A new study created in memory with name: shallow_reg
[I 2026-08-10 01:39:19,942] Trial 0 finished with value: 0.6233174204826355 and parameters: {'hidden_size': 128, 'lr': 0.00859001743156823, 'batch_size': 16, 'weight_decay': 7.262266887516058e-05, 'dropout': 0.05642680822122102}. Best is trial 0 with value: 0.6233174204826355.
[I 2026-08-10 01:39:50,605] Trial 1 finished with value: 0.6178327202796936 and parameters: {'hidden_size': 64, 'lr': 0.007989640849363534, 'batch_size': 16, 'weight_decay': 0.00023431412529330493, 'dropout': 0.23411873181496629}. Best is trial 1 with value: 0.6178327202796936.
[I 2026-08-10 01:40:28,065] Trial 2 finished with value: 0.6284648776054382 and parameters: {'hidden_size': 128, 'lr': 0.0015931789558744547, 'batch_size': 32, 'weight_decay': 6.369613727304984e-06, 'dropout': 0.34130388690384456}. Best is trial 1 with value: 0.6178327202796936.
[I 2026-08-10 01:41:03,638] Trial 3 finished with value: 0.6299251914024353 and pa


ShallowLSTM best val MAE: 0.6178
Best params: {'hidden_size': 64, 'lr': 0.007989640849363534, 'batch_size': 16, 'weight_decay': 0.00023431412529330493, 'dropout': 0.23411873181496629}
Optuna time: 18.4 min


## Model B: StackedLSTM (L=4)

In [ ]:
def make_stacked_fn(params, fi, num_layers=4):
    model = StackedLSTM(
        input_size=len(selected_indices),
        hidden_size=params['hidden_size'],
        num_layers=num_layers,
        num_targets=len(PREDICTION_TARGETS),
        dropout=params.get('dropout', 0.2),
        static_dim=STATIC_DIM_IMP,
    ).to(DEVICE)
    init_lstm_submodule(model)
    return model

_t0 = _time.time()
study_B = run_optuna(
    "stacked4_reg",
    get_hpo_objective_mtl(make_stacked_fn, selected_indices),
    n_trials=OPTUNA_TRIALS,
    seed_trials=[{
        'hidden_size': 128, 'lr': 0.00859001743156823, 'batch_size': 16,
        'weight_decay': 7.262266887516058e-05, 'dropout': 0.05642680822122102,
    }],
)
print(f"\nStackedLSTM L=4 best val MAE: {study_B.best_value:.4f}")
print(f"Best params: {study_B.best_trial.params}")
print(f"Optuna time: {(_time.time()-_t0)/60:.1f} min")

[I 2026-08-10 01:56:57,212] A new study created in memory with name: stacked4_reg
[I 2026-08-10 01:59:06,737] Trial 0 finished with value: 0.6276519894599915 and parameters: {'hidden_size': 128, 'lr': 0.00859001743156823, 'batch_size': 16, 'weight_decay': 7.262266887516058e-05, 'dropout': 0.05642680822122102}. Best is trial 0 with value: 0.6276519894599915.
[I 2026-08-10 01:59:39,509] Trial 1 finished with value: 0.6376740336418152 and parameters: {'hidden_size': 32, 'lr': 0.00010868410525237324, 'batch_size': 64, 'weight_decay': 1.7770496464630108e-06, 'dropout': 0.024456039169275356}. Best is trial 0 with value: 0.6276519894599915.
[I 2026-08-10 02:02:32,342] Trial 2 finished with value: 0.6260287165641785 and parameters: {'hidden_size': 64, 'lr': 0.00016944350254862852, 'batch_size': 16, 'weight_decay': 3.032140393784631e-05, 'dropout': 0.05693633940728743}. Best is trial 2 with value: 0.6260287165641785.
[I 2026-08-10 02:03:19,950] Trial 3 finished with value: 0.6233630776405334 an


StackedLSTM L=4 best val MAE: 0.6234
Best params: {'hidden_size': 64, 'lr': 0.005960287286084612, 'batch_size': 32, 'weight_decay': 0.00021726664099383768, 'dropout': 0.35347287196824356}
Optuna time: 29.3 min


## Model C: MultiFreqLSTM (Daily and Monthly only)


In [ ]:
OPTUNA_TRIALS = 8

def make_mf_c_fn_for_optuna(params):
    model = MultiFreqLSTM_NoQuarterly(
        n_daily=len(daily_indices),
        n_monthly=len(monthly_indices) + len(quarterly_indices),
        n_static=STATIC_DIM_IMP,
        d_daily=64, d_monthly=32,
        n_layers_daily=params.get('n_layers_daily', 2),
        n_layers_monthly=params.get('n_layers_monthly', 1),
        num_targets=len(PREDICTION_TARGETS),
        dropout=params.get('dropout', 0.1),
    ).to(DEVICE)
    init_lstm_submodule(model)
    return model

def get_hpo_objective_mf_gdp():
    """IT_GDP is folded into the monthly stream"""
    fold_local = folds[-1]
    tr_df, va_df, val_year, _ = fold_local
    tr_mtl = extract_mtl_df(tr_df)
    va_mtl = extract_mtl_df(va_df)
    monthly_gdp_idx = monthly_indices + quarterly_indices
    daily_sc = make_robust_scaler(tr_mtl, daily_indices)
    monthly_sc = make_robust_scaler(tr_mtl, monthly_gdp_idx)
    stat_sc = fit_static_scaler(tr_mtl, NEW_STATIC_SLICE)
    tr_mtl_s = apply_static_scaler(tr_mtl, stat_sc, NEW_STATIC_SLICE)
    va_mtl_s = apply_static_scaler(va_mtl, stat_sc, NEW_STATIC_SLICE)

    def objective(trial):
        params = dict(
            batch_size=trial.suggest_categorical("batch_size", [16, 32, 64]),
            lr=trial.suggest_float("lr", 1e-4, 1e-2, log=True),
            weight_decay=trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
            dropout=trial.suggest_float("dropout", 0.1, 0.5),
            n_layers_daily=2,
            n_layers_monthly=1,
        )
        tr_ds = MultiFreqDataset(tr_mtl_s, daily_sc, monthly_sc, None, None,
                                 daily_indices, monthly_gdp_idx, [])
        va_ds = MultiFreqDataset(va_mtl_s, daily_sc, monthly_sc, None, None,
                                 daily_indices, monthly_gdp_idx, [])
        tr_loader = DataLoader(tr_ds, batch_size=params['batch_size'], shuffle=True,  collate_fn=collate_fn_multifreq)
        va_loader = DataLoader(va_ds, batch_size=params['batch_size'], shuffle=False, collate_fn=collate_fn_multifreq)
        model = make_mf_c_fn_for_optuna(params)
        optimizer = torch.optim.Adam(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])
        try:
            _, best_metric, _ = train_and_eval_mf(
                model, tr_loader, va_loader, optimizer,
                trial=trial, step_offset=0,
            )
        except optuna.exceptions.TrialPruned:
            raise
        except Exception as e:
            print(f"Trial failed: {type(e).__name__}: {e}")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            return float('inf')
        return best_metric
    return objective

_t0 = _time.time()
study_C = run_optuna(
    "multifreq_c_reg",
    get_hpo_objective_mf_gdp(),
    n_trials=OPTUNA_TRIALS,
)
print(f"\nMultiFreqLSTM best val MAE: {study_C.best_value:.4f}")
print(f"Best params: {study_C.best_trial.params}")
print(f"Optuna time: {(_time.time()-_t0)/60:.1f} min")

[I 2026-08-10 02:26:17,585] A new study created in memory with name: multifreq_c_reg
[I 2026-08-10 02:42:24,288] Trial 0 finished with value: 0.6389617323875427 and parameters: {'batch_size': 64, 'lr': 0.0006231124728222151, 'weight_decay': 2.9476022791259854e-06, 'dropout': 0.14726412855465765}. Best is trial 0 with value: 0.6389617323875427.
[I 2026-08-10 03:00:04,035] Trial 1 finished with value: 0.6376804709434509 and parameters: {'batch_size': 16, 'lr': 0.00011034445641914266, 'weight_decay': 2.760533001247873e-05, 'dropout': 0.2527634919759283}. Best is trial 1 with value: 0.6376804709434509.
[I 2026-08-10 03:17:56,491] Trial 2 finished with value: 0.6373912692070007 and parameters: {'batch_size': 32, 'lr': 0.00010399076817221948, 'weight_decay': 2.372844478986949e-06, 'dropout': 0.10132937748714746}. Best is trial 2 with value: 0.6373912692070007.
[I 2026-08-10 04:06:45,773] Trial 3 finished with value: 0.6221242547035217 and parameters: {'batch_size': 64, 'lr': 0.00115928213319


MultiFreqLSTM best val MAE: 0.6221
Best params: {'batch_size': 64, 'lr': 0.0011592821331991162, 'weight_decay': 0.0006989677248827552, 'dropout': 0.13013955500381963}
Optuna time: 214.1 min


# Evaluations

## Seeds run on test (2024+2025) for each model

In [ ]:
def run_n_seeded_final_mtl(make_model_fn, params, n_seeds=N_FINAL_SEEDS):
    """Train `n_seeds` independent models on train/val (train<=2022, val=2023), test on 2024+2025."""
    results = {t: [] for t in PREDICTION_TARGETS + ["Overall"]}
    histories = []
    best_state_dicts = []
    test_res_per_seed = []
    for s in range(n_seeds):
        np.random.seed(SEED + s)
        torch.manual_seed(SEED + s)
        model, tr_loader, va_loader, te_loader = make_model_fn(params)
        optimizer = torch.optim.Adam(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])
        model, best_val, best_epoch = train_and_eval_mtl(model, tr_loader, va_loader, optimizer)
        test_res = evaluate_mtl_loader(model, te_loader)
        for k, v in test_res.items():
            results[k].append(v)
        test_res_per_seed.append(test_res)
    return results, test_res_per_seed

print("Multi-seed helper ready.")

Multi-seed helper ready.


In [ ]:
class _OptunaStudyStub:
    def __init__(self, best_value, best_params):
        self.best_value = float(best_value)
        self.best_trial = type('TrialStub', (), {'params': dict(best_params)})()

_OPTUNA_RESULTS_A = {
    'best_value': 0.6178327202796936,
    'best_params': {
        'hidden_size': 64, 'lr': 0.007989640849363534, 'batch_size': 16,
        'weight_decay': 0.00023431412529330493, 'dropout': 0.23411873181496629,
    },
}
_OPTUNA_RESULTS_B = {
    'best_value': 0.6233630776405334,
    'best_params': {
        'hidden_size': 64, 'lr': 0.005960287286084612, 'batch_size': 32,
        'weight_decay': 0.00021726664099383768, 'dropout': 0.35347287196824356,
    },
}
_OPTUNA_RESULTS_C = {
    'best_value': 0.6221242547035217,
    'best_params': {
        'batch_size': 64, 'lr': 0.0011592821331991162,
        'weight_decay': 0.0006989677248827552, 'dropout': 0.13013955500381963,
        'n_layers_daily': 2, 'n_layers_monthly': 1,
    },
}

if 'study_A' not in globals():
    study_A = _OptunaStudyStub(_OPTUNA_RESULTS_A['best_value'], _OPTUNA_RESULTS_A['best_params'])
if 'study_B' not in globals():
    study_B = _OptunaStudyStub(_OPTUNA_RESULTS_B['best_value'], _OPTUNA_RESULTS_B['best_params'])
if 'study_C' not in globals():
    study_C = _OptunaStudyStub(_OPTUNA_RESULTS_C['best_value'], _OPTUNA_RESULTS_C['best_params'])

params_A = study_A.best_trial.params
params_B = study_B.best_trial.params
params_C = study_C.best_trial.params
print(f"ShallowLSTM    best val MAE: {study_A.best_value:.4f}")
print(f"ShallowLSTM    best params: {params_A}")
print(f"StackedLSTM L=4 best val MAE: {study_B.best_value:.4f}")
print(f"StackedLSTM L=4 best params: {params_B}")
print(f"MultiFreqLSTM best val MAE: {study_C.best_value:.4f}")
print(f"MultiFreqLSTM best params: {params_C}")

ShallowLSTM    best val MAE: 0.6178
ShallowLSTM    best params: {'hidden_size': 64, 'lr': 0.007989640849363534, 'batch_size': 16, 'weight_decay': 0.00023431412529330493, 'dropout': 0.23411873181496629}
StackedLSTM L=4 best val MAE: 0.6234
StackedLSTM L=4 best params: {'hidden_size': 64, 'lr': 0.005960287286084612, 'batch_size': 32, 'weight_decay': 0.00021726664099383768, 'dropout': 0.35347287196824356}
MultiFreqLSTM best val MAE: 0.6221
MultiFreqLSTM best params: {'batch_size': 64, 'lr': 0.0011592821331991162, 'weight_decay': 0.0006989677248827552, 'dropout': 0.13013955500381963, 'n_layers_daily': 2, 'n_layers_monthly': 1}


In [ ]:
_STUDY_STUBS = {
    "study_A": _OPTUNA_RESULTS_A,
    "study_B": _OPTUNA_RESULTS_B,
    "study_C": _OPTUNA_RESULTS_C,
}
for _sn in _STUDY_STUBS:
    if _sn not in globals():
        _res = _STUDY_STUBS[_sn]
        globals()[_sn] = _OptunaStudyStub(_res['best_value'], _res['best_params'])
        print(f"NOTE: {_sn} not in globals; using hardcoded Optuna stub.")
params_A = study_A.best_trial.params
params_B = study_B.best_trial.params
params_C = study_C.best_trial.params
print(f"ShallowLSTM   best val MAE: {study_A.best_value:.4f}")
print(f"ShallowLSTM   best params: {params_A}")
print(f"StackedLSTM L=4 best val MAE: {study_B.best_value:.4f}")
print(f"StackedLSTM L=4 best params: {params_B}")
print(f"MultiFreqLSTM best val MAE: {study_C.best_value:.4f}")
print(f"MultiFreqLSTM best params: {params_C}")

ShallowLSTM   best val MAE: 0.6178
ShallowLSTM   best params: {'hidden_size': 64, 'lr': 0.007989640849363534, 'batch_size': 16, 'weight_decay': 0.00023431412529330493, 'dropout': 0.23411873181496629}
StackedLSTM L=4 best val MAE: 0.6234
StackedLSTM L=4 best params: {'hidden_size': 64, 'lr': 0.005960287286084612, 'batch_size': 32, 'weight_decay': 0.00021726664099383768, 'dropout': 0.35347287196824356}
MultiFreqLSTM best val MAE: 0.6221
MultiFreqLSTM best params: {'batch_size': 64, 'lr': 0.0011592821331991162, 'weight_decay': 0.0006989677248827552, 'dropout': 0.13013955500381963, 'n_layers_daily': 2, 'n_layers_monthly': 1}


In [ ]:
params_A = study_A.best_trial.params
print(f"ShallowLSTM final params: {params_A}")
def make_A(params):
    return make_shallow_mtl(train_data, val_data, params, selected_indices)
_t0 = _time.time()
results_A, test_res_A = run_n_seeded_final_mtl(make_A, params_A, n_seeds=N_FINAL_SEEDS)
print(f"\nShallowLSTM final test (n={N_FINAL_SEEDS} seeds):")
for k in PREDICTION_TARGETS + ["Overall"]:
    vals = results_A[k]
    print(f"  {k:12s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")
print(f"Time: {(_time.time()-_t0)/60:.1f} min")

ShallowLSTM final params: {'hidden_size': 64, 'lr': 0.007989640849363534, 'batch_size': 16, 'weight_decay': 0.00023431412529330493, 'dropout': 0.23411873181496629}

ShallowLSTM final test (n=5 seeds):
  EBITDA      : 0.4234 ± 0.0152
  Net_Income  : 0.6908 ± 0.0150
  ROA         : 0.6551 ± 0.0060
  Overall     : 0.5899 ± 0.0113
Time: 4.4 min


In [ ]:
params_B = study_B.best_trial.params
print(f"StackedLSTM L=4 final params: {params_B}")
def make_B(params):
    return make_stacked_mtl(train_data, val_data, params, selected_indices, num_layers=4)
_t0 = _time.time()
results_B, test_res_B = run_n_seeded_final_mtl(make_B, params_B, n_seeds=N_FINAL_SEEDS)
print(f"\nStackedLSTM L=4 final test (n={N_FINAL_SEEDS} seeds):")
for k in PREDICTION_TARGETS + ["Overall"]:
    vals = results_B[k]
    print(f"  {k:12s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")
print(f"Time: {(_time.time()-_t0)/60:.1f} min")

StackedLSTM L=4 final params: {'hidden_size': 64, 'lr': 0.005960287286084612, 'batch_size': 32, 'weight_decay': 0.00021726664099383768, 'dropout': 0.35347287196824356}

StackedLSTM L=4 final test (n=5 seeds):
  EBITDA      : 0.4073 ± 0.0035
  Net_Income  : 0.6798 ± 0.0072
  ROA         : 0.6498 ± 0.0060
  Overall     : 0.5791 ± 0.0052
Time: 5.4 min


In [ ]:
def make_multifreq_gdp_loaders_for_fold(tr_df, va_df, params):
    """Model C loader builder: daily + monthly only.
    IT_GDP is folded into the monthly stream (forward-filled at monthly
    cadence); no separate quarterly MLP is used.
    """
    monthly_gdp_idx = monthly_indices + quarterly_indices
    tr_mtl = extract_mtl_df(tr_df)
    va_mtl = extract_mtl_df(va_df)
    te_mtl = extract_mtl_df(test_data)
    daily_sc = make_robust_scaler(tr_mtl, daily_indices)
    monthly_sc = make_robust_scaler(tr_mtl, monthly_gdp_idx)
    stat_sc = fit_static_scaler(tr_mtl, NEW_STATIC_SLICE)
    tr_mtl_s = apply_static_scaler(tr_mtl, stat_sc, NEW_STATIC_SLICE)
    va_mtl_s = apply_static_scaler(va_mtl, stat_sc, NEW_STATIC_SLICE)
    te_mtl_s = apply_static_scaler(te_mtl, stat_sc, NEW_STATIC_SLICE)
    tr_ds = MultiFreqDataset(tr_mtl_s, daily_sc, monthly_sc, None, None,
                             daily_indices, monthly_gdp_idx, [])
    va_ds = MultiFreqDataset(va_mtl_s, daily_sc, monthly_sc, None, None,
                             daily_indices, monthly_gdp_idx, [])
    te_ds = MultiFreqDataset(te_mtl_s, daily_sc, monthly_sc, None, None,
                             daily_indices, monthly_gdp_idx, [])
    tr_loader = DataLoader(tr_ds, batch_size=params['batch_size'], shuffle=True,  collate_fn=collate_fn_multifreq,
                           num_workers=2, pin_memory=True, persistent_workers=True)
    va_loader = DataLoader(va_ds, batch_size=params['batch_size'], shuffle=False, collate_fn=collate_fn_multifreq,
                           num_workers=2, pin_memory=True, persistent_workers=True)
    te_loader = DataLoader(te_ds, batch_size=params['batch_size'], shuffle=False, collate_fn=collate_fn_multifreq,
                           num_workers=2, pin_memory=True, persistent_workers=True)
    model = MultiFreqLSTM_NoQuarterly(
        n_daily=len(daily_indices),
        n_monthly=len(monthly_gdp_idx),
        n_static=STATIC_DIM_IMP,
        d_daily=64, d_monthly=32,
        n_layers_daily=params.get('n_layers_daily', 2),
        n_layers_monthly=params.get('n_layers_monthly', 1),
        num_targets=len(PREDICTION_TARGETS),
        dropout=params.get('dropout', 0.1),
    ).to(DEVICE)
    init_lstm_submodule(model)
    return model, tr_loader, va_loader, te_loader


In [ ]:
def run_n_seeded_final_mf_loader(make_model_fn, params, n_seeds=2):
    """Train `n_seeds` independent Model C (MultiFreq-style) models, evaluate on test set."""
    results = {t: [] for t in PREDICTION_TARGETS + ["Overall"]}
    for s in range(n_seeds):
        np.random.seed(SEED + s); torch.manual_seed(SEED + s)
        model, tr_loader, va_loader, te_loader = make_model_fn(train_data, val_data, params)
        optimizer = torch.optim.Adam(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])
        model, _, _ = train_and_eval_mf(model, tr_loader, va_loader, optimizer)
        test_res = evaluate_mf_loader(model, te_loader)
        for k, v in test_res.items():
            results[k].append(v)
    return results

params_C = study_C.best_trial.params
print(f"MultiFreqLSTM final params: {params_C}")
N_FINAL_SEEDS_C = 2
_t0 = _time.time()
results_C = run_n_seeded_final_mf_loader(make_multifreq_gdp_loaders_for_fold, params_C, n_seeds=N_FINAL_SEEDS_C)
print(f"\nMultiFreqLSTM final test (n={N_FINAL_SEEDS_C} seeds):")
for k in PREDICTION_TARGETS + ["Overall"]:
    vals = results_C[k]
    print(f"  {k:12s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")
print(f"Time: {(_time.time()-_t0)/60:.1f} min")

MultiFreqLSTM final params: {'batch_size': 64, 'lr': 0.0011592821331991162, 'weight_decay': 0.0006989677248827552, 'dropout': 0.13013955500381963, 'n_layers_daily': 2, 'n_layers_monthly': 1}

MultiFreqLSTM final test (n=2 seeds):
  EBITDA      : 0.4229 ± 0.0154
  Net_Income  : 0.6826 ± 0.0087
  ROA         : 0.6499 ± 0.0022
  Overall     : 0.5853 ± 0.0088
Time: 80.2 min


## Per-Fold Evaluation

In [ ]:
N_SEEDS_PER_FOLD = 3

def run_per_fold_final_mtl(make_model_fn, params, n_seeds=N_SEEDS_PER_FOLD, collect_preds=True):
    """For each fold, train K seeds with the best Optuna params and evaluate on test set.
    Returns a nested dict: per_fold[fold_idx] = {
        'val_year': int, 'seed_maes': [list of per-seed test MAE dicts],
        'preds_untrained': {seed_idx: array (real space)},
        'preds_trained':   {seed_idx: array (real space)},
        'targets':          array (real space),
    }"""
    per_fold = {}
    for fold_idx, (tr_df, va_df, val_year, fi) in enumerate(folds):
        seed_maes = []
        preds_u_dict = {}
        preds_t_dict = {}
        targets_arr = None
        for s in range(n_seeds):
            np.random.seed(SEED + s); torch.manual_seed(SEED + s)
            # Untrained baseline predictions
            model_u, tr_loader_u, va_loader_u, te_loader_u = make_model_fn(params, tr_df, va_df)
            yp_u, yt = collect_predictions_mtl(model_u, te_loader_u) if collect_preds else (None, None)
            del model_u
            # Train
            np.random.seed(SEED + s); torch.manual_seed(SEED + s)
            model, tr_loader, va_loader, te_loader = make_model_fn(params, tr_df, va_df)
            optimizer = torch.optim.Adam(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])
            model, _, _ = train_and_eval_mtl(model, tr_loader, va_loader, optimizer)
            test_res = evaluate_mtl_loader(model, te_loader)
            seed_maes.append(test_res)
            if collect_preds:
                yp_t, yt_chk = collect_predictions_mtl(model, te_loader)
                targets_arr = yt_chk
                preds_u_dict[s] = yp_u
                preds_t_dict[s] = yp_t
            del model
        per_fold[fold_idx] = {
            'val_year': int(val_year),
            'seed_maes': seed_maes,
            'preds_untrained': preds_u_dict,
            'preds_trained':   preds_t_dict,
            'targets':          targets_arr,
        }
        maes_overall = [m['Overall'] for m in seed_maes]
        print(f"  Fold {fi} (val={val_year}): "
              f"Overall MAE = {np.mean(maes_overall):.4f} ± {np.std(maes_overall):.4f}")
    return per_fold

In [ ]:
def make_A_for_fold(params, tr_df, va_df):
    return make_shallow_mtl(tr_df, va_df, params, selected_indices)

def make_B_for_fold(params, tr_df, va_df):
    return make_stacked_mtl(tr_df, va_df, params, selected_indices, num_layers=4)

In [ ]:
print("ShallowLSTM per-fold test:")
_t0 = _time.time()
per_fold_A = run_per_fold_final_mtl(make_A_for_fold, params_A, n_seeds=2)  # 2 seeds for A
print(f"Time: {(_time.time()-_t0)/60:.1f} min")

ShallowLSTM per-fold test:
  Fold 1 (val=2019): Overall MAE = 0.6320 ± 0.0274
  Fold 2 (val=2020): Overall MAE = 0.5971 ± 0.0025
  Fold 3 (val=2021): Overall MAE = 0.5958 ± 0.0027
  Fold 4 (val=2022): Overall MAE = 0.5830 ± 0.0025
  Fold 5 (val=2023): Overall MAE = 0.5980 ± 0.0136
Time: 9.4 min


In [ ]:
print("StackedLSTM L=4 per-fold test:")
_t0 = _time.time()
per_fold_B = run_per_fold_final_mtl(make_B_for_fold, params_B, n_seeds=2)  # 2 seeds for B
print(f"Time: {(_time.time()-_t0)/60:.1f} min")

StackedLSTM L=4 per-fold test:
  Fold 1 (val=2019): Overall MAE = 0.5823 ± 0.0030
  Fold 2 (val=2020): Overall MAE = 0.5840 ± 0.0034
  Fold 3 (val=2021): Overall MAE = 0.6018 ± 0.0212
  Fold 4 (val=2022): Overall MAE = 0.5825 ± 0.0087
  Fold 5 (val=2023): Overall MAE = 0.5822 ± 0.0048
Time: 11.9 min


In [ ]:
def run_per_fold_final_mf_loader(make_model_fn, params, n_seeds=1, collect_preds=True):
    """Model C version of per-fold evaluation (MultiFreq-style)."""
    per_fold = {}
    for fold_idx, (tr_df, va_df, val_year, fi) in enumerate(folds):
        seed_maes = []
        preds_u_dict = {}
        preds_t_dict = {}
        targets_arr = None
        for s in range(n_seeds):
            np.random.seed(SEED + s); torch.manual_seed(SEED + s)
            model_u, _, _, te_loader_u = make_model_fn(tr_df, va_df, params)
            yp_u, yt = collect_predictions_mf(model_u, te_loader_u) if collect_preds else (None, None)
            del model_u
            np.random.seed(SEED + s); torch.manual_seed(SEED + s)
            model, tr_loader, va_loader, te_loader = make_model_fn(tr_df, va_df, params)
            optimizer = torch.optim.Adam(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])
            model, _, _ = train_and_eval_mf(model, tr_loader, va_loader, optimizer)
            test_res = evaluate_mf_loader(model, te_loader)
            seed_maes.append(test_res)
            if collect_preds:
                yp_t, yt_chk = collect_predictions_mf(model, te_loader)
                targets_arr = yt_chk
                preds_u_dict[s] = yp_u
                preds_t_dict[s] = yp_t
            del model
        per_fold[fold_idx] = {
            'val_year': int(val_year),
            'seed_maes': seed_maes,
            'preds_untrained': preds_u_dict,
            'preds_trained':   preds_t_dict,
            'targets':          targets_arr,
        }
        maes_overall = [m['Overall'] for m in seed_maes]
        print(f"  Fold {fi} (val={val_year}): "
              f"Overall MAE = {np.mean(maes_overall):.4f} ± {np.std(maes_overall):.4f}")
    return per_fold

if not params_C:
    raise RuntimeError(
        "Model C Optuna results are empty. Run the Model C Optuna cell and fill"
        "_OPTUNA_RESULTS_C before running this cell."
    )
print("MultiFreqLSTM per-fold test:")
_t0 = _time.time()
per_fold_C = run_per_fold_final_mf_loader(make_multifreq_gdp_loaders_for_fold, params_C, n_seeds=1)  # 1 seed because takes longer
print(f"Time: {(_time.time()-_t0)/60:.1f} min")

MultiFreqLSTM per-fold test:
  Fold 1 (val=2019): Overall MAE = 0.5825 ± 0.0000
  Fold 2 (val=2020): Overall MAE = 0.5930 ± 0.0000
  Fold 3 (val=2021): Overall MAE = 0.5778 ± 0.0000
  Fold 4 (val=2022): Overall MAE = 0.6152 ± 0.0000
  Fold 5 (val=2023): Overall MAE = 0.5765 ± 0.0000
Time: 182.4 min


In [5]:
SINGLE_SPLIT = {
    "ShallowLSTM": {
        "n_seeds": 5,
        "EBITDA":     (0.4234, 0.0152),
        "Net_Income": (0.6908, 0.0150),
        "ROA":        (0.6551, 0.0060),
        "Overall":    (0.5899, 0.0113),
    },
    "StackedLSTM L=4": {
        "n_seeds": 5,
        "EBITDA":     (0.4073, 0.0035),
        "Net_Income": (0.6798, 0.0072),
        "ROA":        (0.6498, 0.0060),
        "Overall":    (0.5791, 0.0052),
    },
    "MultiFreqLSTM": {
        "n_seeds": 2,
        "EBITDA":     (0.4229, 0.0154),
        "Net_Income": (0.6826, 0.0087),
        "ROA":        (0.6499, 0.0022),
        "Overall":    (0.5853, 0.0088),
    },
}

PER_FOLD = {
    "ShallowLSTM":      {2019: 0.6320, 2020: 0.5971, 2021: 0.5958, 2022: 0.5830, 2023: 0.5980},
    "StackedLSTM L=4":  {2019: 0.5823, 2020: 0.5840, 2021: 0.6018, 2022: 0.5825, 2023: 0.5822},
    "MultiFreqLSTM":    {2019: 0.5825, 2020: 0.5930, 2021: 0.5778, 2022: 0.6152, 2023: 0.5765},
}

OPTUNA_VAL = {
    "ShallowLSTM":     0.6178327202796936,
    "StackedLSTM L=4": 0.6233630776405334,
    "MultiFreqLSTM":   0.6221242547035217,
}

BEST_PARAMS = {
    "ShallowLSTM": {
        'hidden_size':   64,
        'lr':            0.007989640849363534,
        'batch_size':    16,
        'weight_decay':  0.00023431412529330493,
        'dropout':       0.23411873181496629,
    },
    "StackedLSTM L=4": {
        'hidden_size':   64,
        'lr':            0.005960287286084612,
        'batch_size':    32,
        'weight_decay':  0.00021726664099383768,
        'dropout':       0.35347287196824356,
    },
    "MultiFreqLSTM": {
        'batch_size':       64,
        'lr':               0.0011592821331991162,
        'weight_decay':     0.0006989677248827552,
        'dropout':          0.13013955500381963,
        'n_layers_daily':   2,
        'n_layers_monthly': 1,
    },
}

def pf_summary(pf):
    """Mean +/- std of per-fold means (std across the 5 fold means, population)."""
    means = list(pf.values())
    return f"{np.mean(means):.4f} \u00b1 {np.std(means):.4f}"

pf_str = {m: pf_summary(PER_FOLD[m]) for m in PER_FOLD}

print("Per-fold test MAE (Overall), per validation year:")
for m, pf in PER_FOLD.items():
    years = sorted(pf.keys())
    print(f"  {m:28s}: " + ", ".join(f"{y}: {pf[y]:.4f}" for y in years))
print()
print("Single-split test MAE (Overall):")
for m, r in SINGLE_SPLIT.items():
    print(f"  {m:28s} (n={r['n_seeds']} seeds): {r['Overall'][0]:.4f} \u00b1 {r['Overall'][1]:.4f}")

Per-fold test MAE (Overall), per validation year:
  ShallowLSTM                 : 2019: 0.6320, 2020: 0.5971, 2021: 0.5958, 2022: 0.5830, 2023: 0.5980
  StackedLSTM L=4             : 2019: 0.5823, 2020: 0.5840, 2021: 0.6018, 2022: 0.5825, 2023: 0.5822
  MultiFreqLSTM               : 2019: 0.5825, 2020: 0.5930, 2021: 0.5778, 2022: 0.6152, 2023: 0.5765

Single-split test MAE (Overall):
  ShallowLSTM                  (n=5 seeds): 0.5899 ± 0.0113
  StackedLSTM L=4              (n=5 seeds): 0.5791 ± 0.0052
  MultiFreqLSTM                (n=2 seeds): 0.5853 ± 0.0088


## Final Comparison Table

In [6]:
summary = pd.DataFrame([
    {
        "Model": m,
        "Best Optuna val MAE": OPTUNA_VAL[m],
        "Test MAE (EBITDA)": f"{SINGLE_SPLIT[m]['EBITDA'][0]:.4f} ± {SINGLE_SPLIT[m]['EBITDA'][1]:.4f}",
        "Test MAE (Net_Income)": f"{SINGLE_SPLIT[m]['Net_Income'][0]:.4f} ± {SINGLE_SPLIT[m]['Net_Income'][1]:.4f}",
        "Test MAE (ROA)": f"{SINGLE_SPLIT[m]['ROA'][0]:.4f} ± {SINGLE_SPLIT[m]['ROA'][1]:.4f}",
        "Test MAE (Overall)": f"{SINGLE_SPLIT[m]['Overall'][0]:.4f} ± {SINGLE_SPLIT[m]['Overall'][1]:.4f}",
        "Per-fold test MAE": pf_str[m],
        "Best params": str(BEST_PARAMS[m]),
    }
    for m in ["ShallowLSTM", "StackedLSTM L=4", "MultiFreqLSTM"]
])
display(summary)

# Best model by single-split headline number
best_idx = summary["Test MAE (Overall)"].apply(lambda s: float(s.split(' ± ')[0])).idxmin()
print(f"\nBest model by single-split test Overall MAE: {summary.loc[best_idx, 'Model']}")
print(f"  Test MAE (Overall): {summary.loc[best_idx, 'Test MAE (Overall)']}")
print(f"  Per-fold test MAE:   {summary.loc[best_idx, 'Per-fold test MAE']}")

# Best model by per-fold test MAE
best_pf_idx = summary["Per-fold test MAE"].apply(lambda s: float(s.split(' ± ')[0])).idxmin()
print(f"\nBest model by per-fold test MAE: {summary.loc[best_pf_idx, 'Model']}")
print(f"  Per-fold test MAE:   {summary.loc[best_pf_idx, 'Per-fold test MAE']}")

,Model,Best Optuna val MAE,Test MAE (EBITDA),Test MAE (Net_Income),Test MAE (ROA),Test MAE (Overall),Per-fold test MAE,Best params
0,ShallowLSTM,0.617833,0.4234 ± 0.0152,0.6908 ± 0.0150,0.6551 ± 0.0060,0.5899 ± 0.0113,0.6012 ± 0.0163,"{'hidden_size': 64, 'lr': 0.007989640849363534..."
1,StackedLSTM L=4,0.623363,0.4073 ± 0.0035,0.6798 ± 0.0072,0.6498 ± 0.0060,0.5791 ± 0.0052,0.5866 ± 0.0076,"{'hidden_size': 64, 'lr': 0.005960287286084612..."
2,MultiFreqLSTM,0.622124,0.4229 ± 0.0154,0.6826 ± 0.0087,0.6499 ± 0.0022,0.5853 ± 0.0088,0.5890 ± 0.0143,"{'batch_size': 64, 'lr': 0.0011592821331991162..."



Best model by single-split test Overall MAE: StackedLSTM L=4
  Test MAE (Overall): 0.5791 ± 0.0052
  Per-fold test MAE:   0.5866 ± 0.0076

Best model by per-fold test MAE: StackedLSTM L=4
  Per-fold test MAE:   0.5866 ± 0.0076


## 2025 as Validation

In [ ]:
TRAIN_MAX_YEAR_FV = 2022
VAL_YEAR_FV      = 2025
TEST_YEARS_FV    = [2023, 2024]

train_fv        = data_df[data_df["year"] <= TRAIN_MAX_YEAR_FV].copy().reset_index(drop=True)
val_fv          = data_df[data_df["year"] == VAL_YEAR_FV].copy().reset_index(drop=True)
test_2023_fv    = data_df[data_df["year"] == 2023].copy().reset_index(drop=True)
test_2024_fv    = data_df[data_df["year"] == 2024].copy().reset_index(drop=True)
test_combined_fv = pd.concat([test_2023_fv, test_2024_fv], ignore_index=True)

print(f"Train (year <= {TRAIN_MAX_YEAR_FV}): {len(train_fv)} samples")
print(f"Val   (year == {VAL_YEAR_FV}): {len(val_fv)} samples")
print(f"Test 2023: {len(test_2023_fv)} samples")
print(f"Test 2024: {len(test_2024_fv)} samples")
print(f"Test 2023+2024 (combined): {len(test_combined_fv)} samples")

mtl_train_fv_raw         = extract_mtl_df(train_fv)
mtl_val_fv_raw           = extract_mtl_df(val_fv)
mtl_test_2023_fv_raw     = extract_mtl_df(test_2023_fv)
mtl_test_2024_fv_raw     = extract_mtl_df(test_2024_fv)
mtl_test_combined_fv_raw = extract_mtl_df(test_combined_fv)

stat_sc_fv = fit_static_scaler(mtl_train_fv_raw, NEW_STATIC_SLICE)
mtl_train_fv         = apply_static_scaler(mtl_train_fv_raw,         stat_sc_fv, NEW_STATIC_SLICE)
mtl_val_fv_s         = apply_static_scaler(mtl_val_fv_raw,           stat_sc_fv, NEW_STATIC_SLICE)
mtl_test_2023_fv     = apply_static_scaler(mtl_test_2023_fv_raw,     stat_sc_fv, NEW_STATIC_SLICE)
mtl_test_2024_fv     = apply_static_scaler(mtl_test_2024_fv_raw,     stat_sc_fv, NEW_STATIC_SLICE)
mtl_test_combined_fv = apply_static_scaler(mtl_test_combined_fv_raw, stat_sc_fv, NEW_STATIC_SLICE)


In [ ]:
def build_loaders_fv(params, num_layers: int):
    """Build train/val/test23/test24/test_combined loaders + StackedLSTM for the FV setup.
    Scalers fit on the FV-train fold only (no leakage from val=2025 or test 2023/2024).
    """
    temp_sc = make_robust_scaler(mtl_train_fv_raw, selected_indices)
    train_ds  = YoYDatasetMTL(mtl_train_fv,         temp_sc, selected_indices)
    val_ds    = YoYDatasetMTL(mtl_val_fv_s,         temp_sc, selected_indices)
    t23_ds    = YoYDatasetMTL(mtl_test_2023_fv,     temp_sc, selected_indices)
    t24_ds    = YoYDatasetMTL(mtl_test_2024_fv,     temp_sc, selected_indices)
    tc_ds     = YoYDatasetMTL(mtl_test_combined_fv, temp_sc, selected_indices)

    bs = params["batch_size"]
    train_loader         = DataLoader(train_ds, batch_size=bs, shuffle=True,  collate_fn=collate_fn_mtl)
    val_loader           = DataLoader(val_ds,   batch_size=bs, shuffle=False, collate_fn=collate_fn_mtl)
    test23_loader        = DataLoader(t23_ds,   batch_size=bs, shuffle=False, collate_fn=collate_fn_mtl)
    test24_loader        = DataLoader(t24_ds,   batch_size=bs, shuffle=False, collate_fn=collate_fn_mtl)
    test_combined_loader = DataLoader(tc_ds,    batch_size=bs, shuffle=False, collate_fn=collate_fn_mtl)

    model = StackedLSTM(
        len(selected_indices), params["hidden_size"],
        num_layers=num_layers, num_targets=len(PREDICTION_TARGETS),
        dropout=params.get("dropout", 0.0), static_dim=STATIC_DIM_IMP,
    ).to(DEVICE)
    init_lstm_submodule(model)
    return model, train_loader, val_loader, test23_loader, test24_loader, test_combined_loader


def eval_mae_per_target(model, loader):
    """Per-target MAE in real (ratio) space, with arcsinh inversion and the ARCSINH_CAP clip."""
    model.eval()
    P, T, M = [], [], []
    with torch.no_grad():
        for bx, by, bmask, bstatic, lengths in loader:
            logits = model(bx.to(DEVICE), lengths, bstatic.to(DEVICE))
            P.append(logits.cpu().numpy()); T.append(by.numpy()); M.append(bmask.numpy())
    if not P:
        return None
    yp = np.vstack(P); yt = np.vstack(T); m = np.vstack(M).astype(bool)
    yp_c = arcsinh_inverse(yp)
    yt_c = arcsinh_inverse(yt)
    res = {}
    all_errs = []
    for i, t in enumerate(PREDICTION_TARGETS):
        mi = m[:, i]
        if mi.sum() == 0:
            continue
        res[t] = float(np.mean(np.abs(yp_c[mi, i] - yt_c[mi, i])))
        all_errs.extend(np.abs(yp_c[mi, i] - yt_c[mi, i]).tolist())
    res["Overall"] = float(np.mean(all_errs)) if all_errs else None
    return res


In [ ]:
FV_DEPTHS = [1, 2, 3, 4]
N_SEEDS_FV = 5

if "study_B" not in globals():
    raise RuntimeError(
        "study_B is not defined. Re-run the StackedLSTM Optuna cell (cell 24) "
        "so params_B reflects the corrected label transform + weight init."
    )
params_B_fv = dict(study_B.best_trial.params)
print(f"Using study_B best params (fixed across depths): {params_B_fv}")

fv_depth_results = []
fv_depth_best_epochs = {}

for nl in FV_DEPTHS:
    for seed_offset in range(N_SEEDS_FV):
        seed = SEED + seed_offset
        np.random.seed(seed)
        torch.manual_seed(seed)

        model, tr_loader, va_loader, t23_loader, t24_loader, tc_loader = \
            build_loaders_fv(params_B_fv, num_layers=nl)
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=params_B_fv["lr"],
            weight_decay=params_B_fv["weight_decay"],
        )

        # Use this notebook's training loop: L1, ReduceLROnPlateau, early stopping,
        # grad_clip=1.0, MAX_EPOCHS=100. Returns best_val_mae (real-space) and best_epoch.
        model, best_val_mae, best_epoch = train_and_eval_mtl(
            model, tr_loader, va_loader, optimizer
        )

        val_mae            = eval_mae_per_target(model, va_loader)
        test23_mae         = eval_mae_per_target(model, t23_loader)
        test24_mae         = eval_mae_per_target(model, t24_loader)
        test_combined_mae  = eval_mae_per_target(model, tc_loader)

        for tgt in PREDICTION_TARGETS + ["Overall"]:
            fv_depth_results.append({
                "num_layers": nl, "seed": seed, "target": tgt,
                "val_2025_mae":      val_mae.get(tgt),
                "test_2023_mae":     test23_mae.get(tgt),
                "test_2024_mae":     test24_mae.get(tgt),
                "test_combined_mae": test_combined_mae.get(tgt),
                "best_val_mae":      best_val_mae,
                "best_epoch":        best_epoch,
            })
        fv_depth_best_epochs[(nl, seed)] = best_epoch
        print(f"[L={nl} seed={seed_offset}] val_2025={val_mae['Overall']:.4f} "
              f"test_2023={test23_mae['Overall']:.4f} test_2024={test24_mae['Overall']:.4f} "
              f"test_combined={test_combined_mae['Overall']:.4f} "
              f"best_val_mae={best_val_mae:.4f} @ ep {best_epoch}")

df_fv_runs = pd.DataFrame(fv_depth_results)
print(f"\n=== Depth sweep ({FV_DEPTHS} layers) \u2014 Train <= 2022, Val = 2025, Test = 2023 & 2024 ===")
print(f"Total runs: {len(df_fv_runs) // 4} (={len(FV_DEPTHS)} depths x {N_SEEDS_FV} seeds)")


In [ ]:
def fv_agg_mean_std(grp):
    metrics = ["val_2025_mae", "test_2023_mae", "test_2024_mae",
               "test_combined_mae", "best_val_mae", "best_epoch"]
    out = [int(len(grp))]
    for m in metrics:
        out.append(float(grp[m].mean()))
        out.append(float(grp[m].std()))
    return out

result = df_fv_runs.groupby(["num_layers", "target"]).apply(fv_agg_mean_std)
df_fv_summary = pd.DataFrame(result.tolist(), index=result.index).reset_index()
df_fv_summary.columns = [
    "num_layers", "target", "n_seeds",
    "val_2025_mae_mean", "val_2025_mae_std",
    "test_2023_mae_mean", "test_2023_mae_std",
    "test_2024_mae_mean", "test_2024_mae_std",
    "test_combined_mae_mean", "test_combined_mae_std",
    "best_val_mae_mean", "best_val_mae_std",
    "best_epoch_mean", "best_epoch_std",
]
display(df_fv_summary.round(4))

val_by_depth       = df_fv_summary[df_fv_summary["target"] == "Overall"].set_index("num_layers")["val_2025_mae_mean"]
test23_by_depth    = df_fv_summary[df_fv_summary["target"] == "Overall"].set_index("num_layers")["test_2023_mae_mean"]
test24_by_depth    = df_fv_summary[df_fv_summary["target"] == "Overall"].set_index("num_layers")["test_2024_mae_mean"]
test_comb_by_depth = df_fv_summary[df_fv_summary["target"] == "Overall"].set_index("num_layers")["test_combined_mae_mean"]

print("\n=== Verdict (FV, depth only, params from study_B) ===")
for metric_name, series in [
    ("Selection (val_2025)", val_by_depth),
    ("Test MAE (2023)",       test23_by_depth),
    ("Test MAE (2024)",       test24_by_depth),
    ("Test MAE (2023+2024)",  test_comb_by_depth),
]:
    best_L = series.idxmin()
    print(f"  {metric_name:25s}: best L={best_L} ({series[best_L]:.4f})")

fig, ax = plt.subplots(1, 4, figsize=(16, 4.5))
xs = np.arange(len(FV_DEPTHS))
width = 0.6
for i, (col, title) in enumerate([
    ("val_2025_mae_mean",     "Val MAE (2025, future)"),
    ("test_2023_mae_mean",    "Test MAE (2023)"),
    ("test_2024_mae_mean",    "Test MAE (2024)"),
    ("test_combined_mae_mean","Test MAE (2023+2024)"),
]):
    means = [df_fv_summary[(df_fv_summary["num_layers"] == L) & (df_fv_summary["target"] == "Overall")][col].iloc[0] for L in FV_DEPTHS]
    stds  = [df_fv_summary[(df_fv_summary["num_layers"] == L) & (df_fv_summary["target"] == "Overall")][col.replace("mean","std")].iloc[0] for L in FV_DEPTHS]
    ax[i].bar(xs, means, width, yerr=stds, capsize=4, color="steelblue", edgecolor="black")
    ax[i].set_xticks(xs); ax[i].set_xticklabels([f"L={L}" for L in FV_DEPTHS])
    ax[i].set_title(title); ax[i].set_ylabel("MAE (real space)")
    ax[i].grid(True, alpha=0.3, axis="y")
fig.suptitle(f"FV depth sweep \u2014 Train <= 2022, Val = 2025, Test = 2023 & 2024  (\u03bc \u00b1 \u03c3, n={N_SEEDS_FV})")
plt.tight_layout()
plt.show()
